# CDC Diabetes Health Indicators: Benchmark and Base Models
Student: Louis Le Forestier and Team
Class: ESILV A4 DIA 3
Course: Machine Learning Final Project


## 1. Environment Setup and Imports

We import standard data science and machine learning libraries.
All experiments use fixed random seeds for strict reproducibility.


In [ ]:
import os
import sys
import hashlib
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.calibration import calibration_curve

# Add parent directory to path so helper modules in src/ can be imported if needed
sys.path.append(os.path.abspath(".."))


## 2. Data Acquisition and Integrity Verification

We download the official CDC Diabetes Health Indicators dataset directly from the UCI Machine Learning Repository.
We verify the number of rows, columns, and SHA-256 checksum to ensure exact provenance.


In [ ]:
UCI_URL = "https://archive.ics.uci.edu/static/public/891/data.csv"
DATA_DIR = os.path.join("..", "data", "raw")
DATA_FILE = os.path.join(DATA_DIR, "cdc_diabetes_raw.csv")

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(DATA_FILE):
    print("Downloading dataset from UCI repository...")
    urllib.request.urlretrieve(UCI_URL, DATA_FILE)
    print("Download complete.")
else:
    print(f"Local file found at: {DATA_FILE}")

# Compute SHA-256 checksum
sha256 = hashlib.sha256()
with open(DATA_FILE, "rb") as f:
    for chunk in iter(lambda: f.read(65536), b""):
        sha256.update(chunk)
checksum = sha256.hexdigest()

df = pd.read_csv(DATA_FILE)
print("Dataset shape:", df.shape)
print("SHA-256 Checksum:", checksum)
print("Target counts (Diabetes_binary):")
print(df["Diabetes_binary"].value_counts(dropna=False))


## 3. Exploratory Data Analysis (EDA)

We inspect the class distribution and summarize demographic and lifestyle variables.
The target has two classes: 0 (no diabetes) and 1 (prediabetes or diabetes).


In [ ]:
# Class distribution percentage
counts = df["Diabetes_binary"].value_counts()
prevalence = counts[1] / len(df) * 100
print(f"Negative class (0 - No diabetes): {counts[0]} ({100 - prevalence:.2f}%)")
print(f"Positive class (1 - Diabetes / Prediabetes): {counts[1]} ({prevalence:.2f}%)")

# Inspecting feature types and missing values
print("Missing values across columns:", df.isnull().sum().sum())
df.describe().T[["mean", "std", "min", "50%", "max"]]


In [ ]:
# Visualizing class balance
plt.figure(figsize=(6, 4))
plt.bar(["No Diabetes (0)", "Diabetes / Prediabetes (1)"], counts.values, color=["steelblue", "salmon"])
plt.ylabel("Number of respondents")
plt.title("Target Distribution (CDC Diabetes Health Indicators)")
plt.grid(True, linestyle=":")
plt.show()


## 4. Feature Definitions and Leakage-Free Splitting

We partition the 21 predictors into two subsets for subsequent comparison:
1. Lifestyle and Demographic variables (9 features): physical activity, fruits, veggies, smoking, alcohol, sex, age, education, income.
2. Full Enriched set (21 features): adds clinical indicators (BMI, high blood pressure, cholesterol, stroke, heart attack, general health, physical health, mental health, difficulty walking) and healthcare access.

We create a stratified 80/20 train/test split. The 20% test set is held out completely.


In [ ]:
LIFESTYLE_VARS = ["PhysActivity", "Fruits", "Veggies", "Smoker", "HvyAlcoholConsump"]
DEMOGRAPHIC_VARS = ["Sex", "Age", "Education", "Income"]
HEALTH_VARS = [
    "BMI", "HighBP", "HighChol", "CholCheck", "Stroke", 
    "HeartDiseaseorAttack", "GenHlth", "MentHlth", "PhysHlth", "DiffWalk"
]
ACCESS_VARS = ["AnyHealthcare", "NoDocbcCost"]

LIFESTYLE_AND_DEMO = LIFESTYLE_VARS + DEMOGRAPHIC_VARS
ALL_PREDICTORS = LIFESTYLE_AND_DEMO + HEALTH_VARS + ACCESS_VARS

# Exclude ID and target from features
X = df[ALL_PREDICTORS].copy()
y = df["Diabetes_binary"].values

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set:     {X_test.shape[0]} rows")


## 5. Main Baseline Model: Standardized Logistic Regression

We fit a standardized logistic regression on the full training set using 5-fold stratified cross-validation.
We evaluate ROC-AUC, Average Precision (PR-AUC), Brier score, and balanced accuracy.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe_baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

scoring_metrics = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "balanced_accuracy": "balanced_accuracy"
}

cv_results = cross_validate(pipe_baseline, X_train, y_train, cv=cv, scoring=scoring_metrics)

print("5-Fold Cross-Validation Baseline Results:")
print(f"ROC-AUC:           {cv_results['test_roc_auc'].mean():.4f} (+/- {cv_results['test_roc_auc'].std():.4f})")
print(f"Average Precision: {cv_results['test_average_precision'].mean():.4f} (+/- {cv_results['test_average_precision'].std():.4f})")
print(f"Balanced Accuracy: {cv_results['test_balanced_accuracy'].mean():.4f} (+/- {cv_results['test_balanced_accuracy'].std():.4f})")


## 6. Model Interpretability: Logistic Regression Odds Ratios

Because logistic regression models the log-odds log(p / (1 - p)) = w^T x + b, the exponentiated coefficients exp(w_j) give the multiplicative change in diabetes odds per standard deviation increase in feature j.


In [ ]:
# Fit on entire training set to extract coefficients
pipe_baseline.fit(X_train, y_train)
lr_model = pipe_baseline.named_steps["clf"]

coefficients = lr_model.coef_[0]
odds_ratios = np.exp(coefficients)

coef_df = pd.DataFrame({
    "Feature": ALL_PREDICTORS,
    "Coefficient": coefficients,
    "Odds_Ratio": odds_ratios
}).sort_values(by="Coefficient", ascending=False)

print("Top 5 Risk Factors (highest odds ratios):")
print(coef_df.head(5).to_string(index=False))

print("
Top 5 Protective Factors (lowest odds ratios):")
print(coef_df.tail(5).to_string(index=False))

# Horizontal bar chart of odds ratios
plt.figure(figsize=(8, 6))
plt.barh(coef_df["Feature"], coef_df["Odds_Ratio"], color="steelblue")
plt.axvline(1.0, color="red", linestyle="--", label="Neutral (OR = 1.0)")
plt.xlabel("Odds Ratio (exp(w))")
plt.title("Feature Impact on Diabetes Odds (Standardized)")
plt.gca().invert_yaxis()
plt.legend()
plt.grid(True, linestyle=":")
plt.show()


## 7. Comparison Across All Simple Base Models

We evaluate and compare simple, interpretable candidate models on the 5-fold cross-validation scheme:
- Logistic Regression (Unweighted)
- Logistic Regression (Balanced class weights)
- Shallow Decision Tree (depth = 3)
- Shallow Decision Tree (depth = 5)
- Gaussian Naive Bayes


In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Logistic Regression (Balanced)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
    ]),
    "Decision Tree (depth 3)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Decision Tree (depth 5)": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Gaussian Naive Bayes": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", GaussianNB())
    ])
}

comparison_results = []

for name, model in models.items():
    res = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring_metrics)
    comparison_results.append({
        "Model": name,
        "ROC-AUC": res["test_roc_auc"].mean(),
        "Average Precision": res["test_average_precision"].mean(),
        "Balanced Accuracy": res["test_balanced_accuracy"].mean()
    })

comparison_df = pd.DataFrame(comparison_results)
print(comparison_df.to_string(index=False))


## 8. Output Risk Probability and Calibration Analysis

Instead of a binary zero/one label, we output the predicted probability P(Y=1 | X).
We compute the Brier score loss and plot the calibration curve to check if predicted risk probabilities correspond to empirical frequencies.


In [ ]:
# Predict probabilities on test set
y_prob_baseline = pipe_baseline.predict_proba(X_test)[:, 1]

brier_baseline = brier_score_loss(y_test, y_prob_baseline)
print(f"Brier score on test set: {brier_baseline:.4f} (closer to 0 is better)")

# Calibration curve (10 bins)
prob_true, prob_pred = calibration_curve(y_test, y_prob_baseline, n_bins=10, strategy="uniform")

plt.figure(figsize=(7, 5))
plt.plot(prob_pred, prob_true, marker="o", label=f"Logistic Regression (Brier: {brier_baseline:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect Calibration")
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Observed Fraction of Positives")
plt.title("Calibration Curve (Risk Score Reliability)")
plt.legend()
plt.grid(True, linestyle=":")
plt.show()


## 9. Decision Threshold Tuning for Clinical Screening

In a screening application, false negatives (missing a diabetic patient) are far more dangerous than false positives.
We evaluate precision, recall, and balanced accuracy across decision thresholds from 0.1 to 0.5.


In [ ]:
thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
threshold_records = []

for t in thresholds:
    y_pred_t = (y_prob_baseline >= t).astype(int)
    rec = recall_score(y_test, y_pred_t)
    prec = precision_score(y_test, y_pred_t, zero_division=0)
    bal_acc = balanced_accuracy_score(y_test, y_pred_t)
    threshold_records.append({
        "Threshold": t,
        "Sensitivity (Recall)": rec,
        "Precision": prec,
        "Balanced Accuracy": bal_acc
    })

t_df = pd.DataFrame(threshold_records)
print(t_df.to_string(index=False))

plt.figure(figsize=(7, 4))
plt.plot(t_df["Threshold"], t_df["Sensitivity (Recall)"], marker="o", label="Sensitivity (Recall)")
plt.plot(t_df["Threshold"], t_df["Precision"], marker="s", label="Precision")
plt.plot(t_df["Threshold"], t_df["Balanced Accuracy"], marker="^", label="Balanced Accuracy")
plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.title("Trade-off Between Sensitivity and Precision Across Thresholds")
plt.grid(True, linestyle=":")
plt.legend()
plt.show()


## 10. Summary and Conclusions

1. The CDC Diabetes Health Indicators dataset contains 253,680 records with a 13.93% prevalence of diabetes/prediabetes.
2. Standardized Logistic Regression provides a strong baseline with an ROC-AUC of approximately 0.825 and an Average Precision of 0.435.
3. The top risk factors identified by odds ratios are General Health, High Blood Pressure, Body Mass Index, and Age.
4. The predicted probabilities are well-calibrated, yielding a low Brier score.
5. In a screening context, lowering the decision threshold to around 0.20 achieves over 75% sensitivity while maintaining reasonable precision.
